# Step 2: train the 125 ADAGE models

Model N (0 to 124) is in group N // 25 and uses seed 171 + N. Each model is saved to
`runs/models/group{g}/model_seed{seed}.npz`, and in the lab's csv format (`runs/models/lab_format/`) for the R
clustering script. Models that are already saved are skipped. On the cluster, `train_models.slurm` runs this notebook
as a job array, one model per task. The last section makes the training checks in thesis Section 4.1 and 4.2.

Settings, and the lab's code.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

N_GROUPS, MODELS_PER_GROUP, BASE_SEED = 5, 25, 171       # 5 groups of 25 models, seeds 171 to 295

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(SEQADAGE, "Py"))
from adage import SeqAdage

EPOCHS = 100
MODELS = "all"  # "all", or a list of model numbers, e.g. range(25) for group 0
FORCE = False   # True = retrain models that are already saved

# as a batch job, array task N trains only model N and skips the checks
IN_BATCH = "SLURM_ARRAY_TASK_ID" in os.environ
if IN_BATCH:
    MODELS = [int(os.environ["SLURM_ARRAY_TASK_ID"])]

Seeds and file paths.

In [ ]:
def group_seed(g, i):
    """Seed of model i of group g (model number N = 25 g + i has seed 171 + N)."""
    return BASE_SEED + g * MODELS_PER_GROUP + i


def model_path(g, seed):
    return os.path.join(RUN_DIR, "models", f"group{g}", f"model_seed{seed}.npz")


def eadage_path(g):
    return os.path.join(RUN_DIR, "eadage", f"eadage_group{g}.npz")


def load_model(path):
    """A saved model (single model or eADAGE) as a dict of arrays."""
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

Train one model with the tuned hyperparameters and save its weights, biases and loss curves.

In [ ]:
def train_and_save(task, hp, force=False):
    import keras
    g, i = divmod(task, MODELS_PER_GROUP)
    seed = group_seed(g, i)
    path = model_path(g, seed)
    if os.path.exists(path) and not force:
        print(f"model {task} (seed {seed}) is already saved, skipping")
        return
    keras.utils.set_random_seed(seed)  # the starting weights; train_model seeds the noise itself
    sa = SeqAdage.SeqAdage(DATA_PATH, seed=seed, enc_dim=hp["units"], kl1=hp["kl1"], kl2=hp["kl2"],
                           act=hp["act1"], act2=hp["act2"], tied=True, epochs=EPOCHS, init=hp["init"],
                           batch_size=hp["bs"], dropout=hp["dropout"], mm=hp["mm"], lr=hp["lr"], v=2)
    start = time.time()
    sa.train_model()
    secs = time.time() - start

    W, eb, db = sa.autoencoder.get_weights()[:3]
    loss, val_loss = np.array(sa.history.history["loss"]), np.array(sa.history.history["val_loss"])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, W=W, eb=eb, db=db, act1=hp["act1"], act2=hp["act2"], loss=loss, val_loss=val_loss,
             seed=seed, group=g, train_seconds=secs)
    lab_dir = os.path.join(RUN_DIR, "models", "lab_format")  # the files the lab's R script reads
    for folder in ["weights", "bias", "loss", "val_loss"]:
        os.makedirs(os.path.join(lab_dir, folder), exist_ok=True)
    sa.save_weights(lab_dir, f"ad_g{g}_{seed}")
    keras.backend.clear_session()
    print(f"model {task} (group {g}, seed {seed}): val_loss {val_loss[-1]:.5f}, {secs / 60:.1f} min")

Train the models.

In [ ]:
with open(os.path.join(RUN_DIR, "tuning", "best_hparams.json")) as f:
    hp = json.load(f)
print("hyperparameters:", hp)
for task in (range(N_GROUPS * MODELS_PER_GROUP) if MODELS == "all" else MODELS):
    train_and_save(task, hp, FORCE)

## Check the training

Plot functions for the checks in thesis Section 4.1 and 4.2. The loss is the one Keras reports: the mean squared error
between the rebuilt and the clean values (from noisy inputs), plus the weight penalties.

In [ ]:
GROUP_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
TRAIN_COLOR, VAL_COLOR = "#2a78d6", "#eb6834"


def load_all_models():
    models = [load_model(model_path(g, group_seed(g, i)))
              for g in range(N_GROUPS) for i in range(MODELS_PER_GROUP)
              if os.path.exists(model_path(g, group_seed(g, i)))]
    print(f"{len(models)} models saved")
    return models


def encode(m, X):
    """Hidden node activations, h = tanh(X W + b), samples x nodes."""
    return np.tanh(X @ m["W"] + m["eb"])


def reconstruct(m, X):
    """What the decoder rebuilds from the activations, using the transposed weights, samples x genes."""
    return np.tanh(encode(m, X) @ m["W"].T + m["db"])


def held_out_samples():
    """The samples Keras held out during training: validation_split=0.1 keeps the last 10% of samples."""
    X = pd.read_csv(DATA_PATH, index_col=0).values.T.astype("float32")  # samples x genes
    return X[int(np.floor(len(X) * 0.9)):]


def save_and_show(fig, name):
    path = os.path.join(RUN_DIR, "training_check", name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.tight_layout()
    fig.savefig(path, dpi=120)
    plt.show()


def plot_loss_curves(models):
    """Training and validation loss of every model, one panel per group."""
    fig, axes = plt.subplots(1, N_GROUPS, figsize=(3.6 * N_GROUPS, 3.4), sharey=True)
    for g, ax in enumerate(axes):
        for m in [m for m in models if int(m["group"]) == g]:
            epochs = np.arange(1, len(m["loss"]) + 1)
            ax.plot(epochs, m["loss"], color=TRAIN_COLOR, lw=1, alpha=0.5)
            ax.plot(epochs, m["val_loss"], color=VAL_COLOR, lw=1, alpha=0.5)
        ax.set(title=f"group {g}", xlabel="epoch", yscale="log")
    axes[0].set_ylabel("loss (MSE + weight penalties)")
    axes[0].plot([], [], color=TRAIN_COLOR, label="training")
    axes[0].plot([], [], color=VAL_COLOR, label="validation")
    axes[0].legend(frameon=False)
    save_and_show(fig, "loss_curves.png")


def plot_training_summary(models):
    """Per model, by group: final validation loss, validation minus training loss, the epoch where the validation
    loss came within 1% of its final value, and training time."""
    group = np.array([int(m["group"]) for m in models])
    final_val = np.array([m["val_loss"][-1] for m in models])
    gap = np.array([m["val_loss"][-1] - m["loss"][-1] for m in models])
    settled = np.array([1 + np.argmax(m["val_loss"] <= m["val_loss"][-1] * 1.01) for m in models])
    minutes = np.array([float(m["train_seconds"]) / 60 for m in models])
    panels = [(final_val, "final validation loss"), (gap, "validation - training loss\n(last epoch)"),
              (settled, "epoch when validation loss got\nwithin 1% of its final value"),
              (minutes, "training time (minutes)")]
    r = np.random.default_rng(0)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    for ax, (values, title) in zip(axes, panels):
        for g in np.unique(group):
            v = values[group == g]
            ax.scatter(g + r.uniform(-0.15, 0.15, len(v)), v, s=14, color=GROUP_COLORS[g])
            ax.hlines(np.median(v), g - 0.3, g + 0.3, color="black", lw=2)  # group median
        ax.set(title=title, xlabel="group", xticks=np.unique(group))
    axes[1].axhline(0, color="gray", lw=1)
    axes[2].set_ylim(0, max(len(m["loss"]) for m in models) + 1)
    save_and_show(fig, "summary.png")
    print(f"final validation loss: {final_val.min():.6f} to {final_val.max():.6f} | "
          f"gap: median {np.median(gap):.5f} | settled: median epoch {np.median(settled):.0f} | "
          f"time: {minutes.min():.0f} to {minutes.max():.0f} min")


def plot_reconstruction(picks, X):
    """How well one model per group rebuilds the held-out samples: per sample, per gene, and every value."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for k, m in enumerate(picks):
        R = reconstruct(m, X)
        per_sample = [np.corrcoef(X[n], R[n])[0, 1] for n in range(len(X))]
        axes[0].hist(per_sample, bins=40, histtype="step", lw=2, color=GROUP_COLORS[k], label=f"seed {int(m['seed'])}")
        axes[1].hist(((R - X) ** 2).mean(0), bins=60, histtype="step", lw=2, color=GROUP_COLORS[k])
    axes[0].set(title="per sample: real vs rebuilt values", xlabel="Pearson correlation", ylabel="samples")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].set(title="per gene: reconstruction error", xlabel="mean squared error", ylabel="genes", yscale="log")
    R = reconstruct(picks[0], X)
    axes[2].hexbin(X.ravel(), R.ravel(), gridsize=60, bins="log", cmap="Blues")
    axes[2].plot([0, 1], [0, 1], color="gray", lw=1)
    axes[2].set(title=f"seed {int(picks[0]['seed'])}: every held-out value", xlabel="real value", ylabel="rebuilt value")
    save_and_show(fig, "reconstruction.png")


def plot_hidden_nodes(picks, X):
    """Activations on the held-out samples: their spread, saturation, and the size of each node's weights."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
    for k, m in enumerate(picks):
        H = encode(m, X)
        norm = np.linalg.norm(m["W"], axis=0)
        kw = dict(histtype="step", lw=2, color=GROUP_COLORS[k])
        axes[0].hist(H.ravel(), bins=60, density=True, label=f"seed {int(m['seed'])}", **kw)
        axes[1].hist((np.abs(H) > 0.99).mean(0), bins=np.linspace(0, 1, 21), **kw)
        axes[2].hist(H.std(0), bins=40, **kw)
        axes[3].hist(norm, bins=40, **kw)
        print(f"seed {int(m['seed'])}: {100 * (norm < 0.15).mean():.0f}% quiet nodes (weight magnitude < 0.15)")
    axes[0].set(title="all activations", xlabel="activation", ylabel="density")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].set(title="per node: fraction of samples\nwhere it is saturated (|h| > 0.99)", xlabel="fraction",
                ylabel="nodes")
    axes[2].set(title="per node: spread of its activation", xlabel="standard deviation over samples", ylabel="nodes")
    axes[3].set(title="per node: magnitude of its weights", xlabel="L2 norm of the weight vector", ylabel="nodes")
    save_and_show(fig, "hidden_nodes.png")

Loss curves and training summary of every saved model.

In [ ]:
if not IN_BATCH:
    models = load_all_models()
    plot_loss_curves(models)
    plot_training_summary(models)

Reconstruction of the held-out samples and the hidden nodes, for one model from each group.

In [ ]:
if not IN_BATCH:
    X_val = held_out_samples()
    picks = [m for m in models if int(m["seed"]) in [group_seed(g, 0) for g in range(N_GROUPS)]]
    plot_reconstruction(picks, X_val)
    plot_hidden_nodes(picks, X_val)